# Modbus HGNN (Clean Colab Notebook)

Efficient reimplementation of the UHG-enhanced HGNN pipeline for modbus traffic embeddings.

**Before running:** set Colab runtime to **GPU** (Runtime → Change runtime type → GPU).

## 1. Setup

In [ ]:
import torch

TORCH_VERSION = torch.__version__.split("+")[0]
CUDA_VERSION = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"

# Install PyG wheels matched to Colab's preinstalled PyTorch/CUDA
!pip install -q torch-scatter torch-sparse torch-geometric -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_VERSION}.html
!pip install -q geoopt umap-learn scikit-learn tqdm matplotlib

In [ ]:
import os
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import geoopt.optim
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler
from torch_scatter import scatter_mean
from tqdm.auto import tqdm
import umap

In [ ]:
@dataclass
class Config:
    DATA_DIR: str = "/content/drive/MyDrive/modbus/"
    ARTIFACT_DIR: str = "/content/drive/MyDrive/hgnn_artifacts/"
    TRAIN_FILE: str = "train_data_balanced_new.csv"
    VAL_FILE: str = "val_data_balanced_new.csv"
    TEST_FILE: str = "test_data_balanced_new.csv"
    data_percentage: float = 0.20
    k: int = 3
    hidden_channels: int = 64
    out_channels: int = 32
    num_layers: int = 3
    lr: float = 0.001
    weight_decay: float = 0.0005
    spread_weight: float = 0.1
    neg_multiplier: int = 2
    num_epochs: int = 100
    patience: int = 15
    log_every: int = 10
    scheduler_step: int = 10
    scheduler_gamma: float = 0.8
    pca_max_components: int = 50
    tsne_subsample: int = 10_000
    dbscan_eps: float = 0.5
    dbscan_min_samples: int = 5
    attack_label: int = 7
    USE_GPU: bool = True
    REBUILD_KNN: bool = False


cfg = Config()

In [ ]:
import os

from google.colab import drive

mount_point = "/content/drive"

if not os.path.ismount(mount_point):
    if os.path.isdir(mount_point) and os.listdir(mount_point):
        print(
            f"Warning: '{mount_point}' exists and is not empty but is not mounted. "
            "Clearing stale contents before mount."
        )
        !rm -rf {mount_point}/*
    drive.mount(mount_point)
else:
    print(f"Google Drive already mounted at {mount_point}. Re-mounting with force_remount=True.")
    drive.mount(mount_point, force_remount=True)

os.makedirs(cfg.ARTIFACT_DIR, exist_ok=True)

if cfg.USE_GPU and torch.cuda.is_available():
    device = torch.device("cuda")
    torch.set_float32_matmul_precision("high")
else:
    device = torch.device("cpu")
    if cfg.USE_GPU:
        print("WARNING: GPU requested but not available. Enable GPU in Colab runtime.")

print(f"Using device: {device}")

## 2. Data loading and preprocessing

Loads train/val/test splits separately, subsamples each split, fits scaler on train only.

In [ ]:
def load_split(filename: str, frac: float, seed: int = 42) -> pd.DataFrame:
    path = os.path.join(cfg.DATA_DIR, filename)
    df = pd.read_csv(path, low_memory=False)
    if frac < 1.0:
        df = df.sample(frac=frac, random_state=seed).reset_index(drop=True)
    return df


def encode_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    non_numeric = out.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric:
        out = pd.get_dummies(out, columns=non_numeric)
    return out


def prepare_features(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame):
    train_x = encode_features(train_df)
    val_x = encode_features(val_df)
    test_x = encode_features(test_df)

    val_x = val_x.reindex(columns=train_x.columns, fill_value=0)
    test_x = test_x.reindex(columns=train_x.columns, fill_value=0)

    train_means = train_x.mean(numeric_only=True)
    train_x = train_x.fillna(train_means)
    val_x = val_x.fillna(train_means)
    test_x = test_x.fillna(train_means)

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_x)
    val_scaled = scaler.transform(val_x)
    test_scaled = scaler.transform(test_x)

    n_components = min(cfg.pca_max_components, train_scaled.shape[1])
    if n_components < train_scaled.shape[1]:
        pca = PCA(n_components=n_components)
        train_scaled = pca.fit_transform(train_scaled)
        val_scaled = pca.transform(val_scaled)
        test_scaled = pca.transform(test_scaled)
        print(f"PCA applied: {n_components} components")
    else:
        pca = None
        print("PCA skipped: feature count below cap")

    return train_scaled, val_scaled, test_scaled, list(train_x.columns), pca


def to_uhg_space(x: np.ndarray) -> np.ndarray:
    return np.concatenate([x, np.ones((x.shape[0], 1), dtype=x.dtype)], axis=1)


train_df = load_split(cfg.TRAIN_FILE, cfg.data_percentage)
val_df = load_split(cfg.VAL_FILE, cfg.data_percentage)
test_df = load_split(cfg.TEST_FILE, cfg.data_percentage)

print(f"Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

train_np, val_np, test_np, feature_columns, pca = prepare_features(train_df, val_df, test_df)
train_uhg = to_uhg_space(train_np)
val_uhg = to_uhg_space(val_np)
test_uhg = to_uhg_space(test_np)

train_features = torch.as_tensor(train_uhg, dtype=torch.float32, device=device)
val_features = torch.as_tensor(val_uhg, dtype=torch.float32, device=device)
test_features = torch.as_tensor(test_uhg, dtype=torch.float32, device=device)

print(f"Feature dims: {train_np.shape[1]} | UHG dims: {train_uhg.shape[1]}")

## 3. KNN graph construction (cached)

Single-pass KNN build per split. Edge indices are saved to `ARTIFACT_DIR` for fast reruns.

In [ ]:
def build_knn_graph(data: np.ndarray, k: int) -> scipy.sparse.csr_matrix:
    return kneighbors_graph(
        data,
        n_neighbors=k,
        mode="connectivity",
        include_self=False,
        n_jobs=-1,
    )


def knn_to_edge_index(knn_graph: scipy.sparse.csr_matrix) -> torch.Tensor:
    coo = knn_graph.tocoo()
    return torch.tensor(np.vstack([coo.row, coo.col]), dtype=torch.long)


import json
import time


def knn_cache_metadata(name: str, data: np.ndarray) -> dict:
    return {
        "name": name,
        "num_nodes": int(data.shape[0]),
        "num_features": int(data.shape[1]),
        "k": cfg.k,
        "data_percentage": cfg.data_percentage,
    }


def load_or_build_edge_index(name: str, data: np.ndarray) -> torch.Tensor:
    cache_path = os.path.join(cfg.ARTIFACT_DIR, f"{name}_edge_index.pt")
    meta_path = os.path.join(cfg.ARTIFACT_DIR, f"{name}_edge_index.meta.json")
    expected_meta = knn_cache_metadata(name, data)

    if os.path.exists(cache_path) and not cfg.REBUILD_KNN:
        if os.path.exists(meta_path):
            with open(meta_path) as f:
                cached_meta = json.load(f)
            if cached_meta != expected_meta:
                print(
                    f"Cache metadata mismatch for {name}: {cached_meta} != {expected_meta}. "
                    "Set cfg.REBUILD_KNN = True to rebuild."
                )
            else:
                t0 = time.perf_counter()
                edge_index = torch.load(cache_path, map_location="cpu")
                elapsed = time.perf_counter() - t0
                print(
                    f"Loaded cached {name} edge index: {edge_index.shape[1]} edges "
                    f"({elapsed:.2f}s)"
                )
                return edge_index.to(device)
        else:
            edge_index = torch.load(cache_path, map_location="cpu")
            print(f"Loaded cached {name} edge index (no metadata file): {edge_index.shape[1]} edges")
            return edge_index.to(device)

    print(f"Building KNN graph for {name} ({data.shape[0]} nodes)...")
    t0 = time.perf_counter()
    knn_graph = build_knn_graph(data, cfg.k)
    edge_index = knn_to_edge_index(knn_graph)
    elapsed = time.perf_counter() - t0
    torch.save(edge_index.cpu(), cache_path)
    with open(meta_path, "w") as f:
        json.dump(expected_meta, f, indent=2)
    print(f"Saved {name} edge index: {edge_index.shape[1]} edges ({elapsed:.1f}s)")
    return edge_index.to(device)


train_edge_index = load_or_build_edge_index("train", train_np)
val_edge_index = load_or_build_edge_index("val", val_np)
test_edge_index = load_or_build_edge_index("test", test_np)

## 4. HGNN model and UHG loss

In [ ]:
def uhg_inner_product(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return torch.sum(a[:, :-1] * b[:, :-1], dim=-1) - a[:, -1] * b[:, -1]


def uhg_norm(a: torch.Tensor) -> torch.Tensor:
    return torch.sum(a[:, :-1] ** 2, dim=-1) - a[:, -1] ** 2


def uhg_quadrance(a: torch.Tensor, b: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    dot_product = uhg_inner_product(a, b)
    denom = torch.clamp((uhg_norm(a) * uhg_norm(b)).abs(), min=eps)
    # Clamp to non-negative: learned embeddings often leave the UHG manifold,
    # which makes raw quadrance negative and can drive total loss below zero.
    return torch.clamp(1 - (dot_product ** 2) / denom, min=0.0)


def sample_negative_edges(num_nodes: int, num_neg: int, device: torch.device, generator: torch.Generator | None = None) -> torch.Tensor:
    if generator is None:
        return torch.randint(0, num_nodes, (2, num_neg), device=device)
    return torch.randint(0, num_nodes, (2, num_neg), device=device, generator=generator)


class GraphSAGE_CustomScatter(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, append_uhg: bool = True):
        super().__init__()
        self.append_uhg = append_uhg
        self.linear = nn.Linear(in_channels * 2, out_channels)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        source_nodes = edge_index[0]
        target_nodes = edge_index[1]
        source_features = x[source_nodes]
        agg = scatter_mean(source_features, target_nodes, dim=0, dim_size=x.size(0))
        out = self.linear(torch.cat([x, agg], dim=1))
        out = F.relu(out)
        if self.append_uhg:
            out = torch.cat([out, torch.ones((out.size(0), 1), device=out.device)], dim=1)
        return out


class HGNN_CustomScatter(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int, num_layers: int):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GraphSAGE_CustomScatter(in_channels, hidden_channels, append_uhg=True))
        for _ in range(num_layers - 2):
            self.convs.append(GraphSAGE_CustomScatter(hidden_channels + 1, hidden_channels, append_uhg=True))
        self.convs.append(GraphSAGE_CustomScatter(hidden_channels + 1, out_channels, append_uhg=False))

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        for conv in self.convs:
            x = conv(x, edge_index)
        return x


def uhg_loss(
    z: torch.Tensor,
    edge_index: torch.Tensor,
    spread_weight: float = 0.1,
    neg_multiplier: int = 2,
    generator: torch.Generator | None = None,
    return_parts: bool = False,
):
    pos_edge_index = edge_index
    if pos_edge_index.size(1) == 0:
        zero = torch.tensor(0.0, device=z.device)
        parts = {"link_loss": zero, "spread_loss": zero, "total_loss": zero}
        return parts if return_parts else zero

    quad = uhg_quadrance(z[pos_edge_index[0]], z[pos_edge_index[1]])
    pos_score = -quad

    num_neg = max(pos_edge_index.size(1) * neg_multiplier, 1)
    neg_edge_index = sample_negative_edges(z.size(0), num_neg, z.device, generator)
    neg_score = -uhg_quadrance(z[neg_edge_index[0]], z[neg_edge_index[1]])

    pos_loss = F.binary_cross_entropy_with_logits(pos_score, torch.ones_like(pos_score))
    neg_loss = F.binary_cross_entropy_with_logits(neg_score, torch.zeros_like(neg_score))
    link_loss = pos_loss + neg_loss
    spread_loss = quad.mean()
    total_loss = link_loss + spread_weight * spread_loss

    if return_parts:
        return {
            "link_loss": link_loss,
            "spread_loss": spread_loss,
            "total_loss": total_loss,
        }
    return total_loss


model = HGNN_CustomScatter(
    in_channels=train_features.shape[1],
    hidden_channels=cfg.hidden_channels,
    out_channels=cfg.out_channels,
    num_layers=cfg.num_layers,
).to(device)

optimizer = geoopt.optim.RiemannianAdam(
    model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=cfg.scheduler_step, gamma=cfg.scheduler_gamma
)

# Fixed generator for deterministic validation loss (neg sampling was causing val swings).
val_loss_generator = torch.Generator(device=device)
val_loss_generator.manual_seed(42)

## 5. Training with validation and early stopping

Full-graph forward and full-graph loss on the training edges.

In [ ]:
@torch.no_grad()
def evaluate(model: nn.Module, features: torch.Tensor, edge_index: torch.Tensor) -> dict:
    model.eval()
    out = model(features, edge_index)
    parts = uhg_loss(
        out,
        edge_index,
        cfg.spread_weight,
        cfg.neg_multiplier,
        generator=val_loss_generator,
        return_parts=True,
    )
    return {k: float(v.item()) for k, v in parts.items()}


def train_with_early_stopping(
    model: nn.Module,
    train_features: torch.Tensor,
    train_edge_index: torch.Tensor,
    val_features: torch.Tensor,
    val_edge_index: torch.Tensor,
    optimizer,
    scheduler,
) -> dict:
    best_val = float("inf")
    best_epoch = 0
    stale_epochs = 0
    checkpoint_path = os.path.join(cfg.ARTIFACT_DIR, "best_model.pt")
    history = {"train_loss": [], "val_loss": [], "val_link_loss": [], "val_spread_loss": []}

    for epoch in tqdm(range(1, cfg.num_epochs + 1), desc="Training"):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        out = model(train_features, train_edge_index)
        loss = uhg_loss(out, train_edge_index, cfg.spread_weight, cfg.neg_multiplier)
        loss.backward()
        optimizer.step()
        scheduler.step()

        train_loss = float(loss.item())
        val_parts = evaluate(model, val_features, val_edge_index)
        # Early-stop on link loss only (BCE); spread term is a regularizer and was
        # causing false "best" checkpoints when quadrance went negative.
        val_link = val_parts["link_loss"]
        val_total = val_parts["total_loss"]
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_total)
        history["val_link_loss"].append(val_link)
        history["val_spread_loss"].append(val_parts["spread_loss"])

        if val_link < best_val:
            best_val = val_link
            best_epoch = epoch
            stale_epochs = 0
            torch.save(
                {
                    "model": model.state_dict(),
                    "epoch": epoch,
                    "val_link_loss": val_link,
                    "val_total_loss": val_total,
                },
                checkpoint_path,
            )
        else:
            stale_epochs += 1

        if epoch == 1 or epoch % cfg.log_every == 0:
            lr = scheduler.get_last_lr()[0]
            print(
                f"Epoch {epoch:3d} | train={train_loss:.4f} | "
                f"val_link={val_link:.4f} | val_total={val_total:.4f} | "
                f"val_spread={val_parts['spread_loss']:.4f} | "
                f"best_link={best_val:.4f} @ {best_epoch} | lr={lr:.6f}"
            )

        if stale_epochs >= cfg.patience:
            print(f"Early stopping at epoch {epoch} (best val_link={best_val:.4f} @ epoch {best_epoch})")
            break

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model"])
    print(
        f"Loaded best checkpoint from epoch {checkpoint['epoch']} "
        f"(val_link={checkpoint['val_link_loss']:.4f}, val_total={checkpoint['val_total_loss']:.4f})"
    )
    return history


history = train_with_early_stopping(
    model,
    train_features,
    train_edge_index,
    val_features,
    val_edge_index,
    optimizer,
    scheduler,
)

## 6. Export embeddings

In [ ]:
model.eval()
with torch.no_grad():
    embeddings = model(train_features, train_edge_index).cpu().numpy()

embeddings_path = os.path.join(cfg.ARTIFACT_DIR, "embeddings.npy")
np.save(embeddings_path, embeddings)
print(f"Saved embeddings: {embeddings.shape} -> {embeddings_path}")

## 7. Visualization and clustering

- PCA and UMAP on full embeddings
- t-SNE on a subsample (10k) for speed
- DBSCAN run once per embedding type
- Cross-check attack label (`source_ip == 7`)

In [ ]:
def apply_dbscan(embeddings: np.ndarray, eps: float, min_samples: int) -> np.ndarray:
    return DBSCAN(eps=eps, min_samples=min_samples).fit_predict(embeddings)


def plot_2d(reduced: np.ndarray, title: str, labels=None):
    plt.figure(figsize=(10, 7))
    if labels is None:
        plt.scatter(reduced[:, 0], reduced[:, 1], s=8, alpha=0.6)
    else:
        plt.scatter(reduced[:, 0], reduced[:, 1], c=labels, cmap="tab20", s=8, alpha=0.6)
        plt.colorbar()
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()


def cluster_distribution_for_label(true_labels, cluster_labels, target_label: int):
    mask = true_labels == target_label
    if mask.sum() == 0:
        print(f"No rows with label {target_label}")
        return
    counts = pd.Series(cluster_labels[mask]).value_counts().sort_values(ascending=False)
    print(counts.head(20))


embeddings_clean = np.nan_to_num(embeddings, nan=0.0)

pca_2d = PCA(n_components=2).fit_transform(embeddings_clean)
plot_2d(pca_2d, "PCA of HGNN Embeddings")

umap_reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    metric="euclidean",
    n_jobs=-1,
    low_memory=True,
    random_state=42,
)
umap_embeddings = umap_reducer.fit_transform(embeddings_clean)
np.save(os.path.join(cfg.ARTIFACT_DIR, "umap_embeddings.npy"), umap_embeddings)
plot_2d(umap_embeddings, "UMAP of HGNN Embeddings")

rng = np.random.default_rng(42)
n_sub = min(cfg.tsne_subsample, embeddings_clean.shape[0])
sub_idx = rng.choice(embeddings_clean.shape[0], size=n_sub, replace=False)
tsne_embeddings = TSNE(
    n_components=2,
    init="pca",
    learning_rate="auto",
    max_iter=1000,
    random_state=42,
).fit_transform(embeddings_clean[sub_idx])
np.save(os.path.join(cfg.ARTIFACT_DIR, "tsne_embeddings_subsample.npy"), tsne_embeddings)
np.save(os.path.join(cfg.ARTIFACT_DIR, "tsne_subsample_idx.npy"), sub_idx)
plot_2d(tsne_embeddings, f"t-SNE subsample ({n_sub} points)")

dbscan_umap_labels = apply_dbscan(umap_embeddings, cfg.dbscan_eps, cfg.dbscan_min_samples)
dbscan_tsne_labels = apply_dbscan(tsne_embeddings, cfg.dbscan_eps, cfg.dbscan_min_samples)
np.save(os.path.join(cfg.ARTIFACT_DIR, "dbscan_umap_labels.npy"), dbscan_umap_labels)
np.save(os.path.join(cfg.ARTIFACT_DIR, "dbscan_tsne_labels.npy"), dbscan_tsne_labels)

plot_2d(umap_embeddings, "UMAP DBSCAN Clusters", labels=dbscan_umap_labels)
plot_2d(tsne_embeddings, "t-SNE DBSCAN Clusters (subsample)", labels=dbscan_tsne_labels)

In [ ]:
if "source_ip" in train_df.columns:
    train_labels = train_df["source_ip"].to_numpy()
    n = min(len(train_labels), len(dbscan_umap_labels))
    train_labels = train_labels[:n]

    print(f"Attack label distribution (source_ip == {cfg.attack_label}) in UMAP DBSCAN clusters:")
    cluster_distribution_for_label(train_labels, dbscan_umap_labels[:n], cfg.attack_label)

    sub_labels = train_labels[sub_idx]
    print(f"\nAttack label distribution (source_ip == {cfg.attack_label}) in t-SNE DBSCAN clusters (subsample):")
    cluster_distribution_for_label(sub_labels, dbscan_tsne_labels, cfg.attack_label)
else:
    print("source_ip column not found; skipping attack cross-check.")

## Option B: subgraph mini-batching (OOM fallback)

If full-graph training runs out of GPU memory, set `USE_SUBGRAPH_BATCHING = True` and rerun the training cell below instead of section 5.

In [ ]:
USE_SUBGRAPH_BATCHING = False
SUBGRAPH_BATCH_SIZE = 4096


def sample_subgraph(num_nodes: int, batch_size: int, edge_index: torch.Tensor):
    node_idx = torch.randperm(num_nodes, device=edge_index.device)[:batch_size]
    node_set = torch.zeros(num_nodes, dtype=torch.bool, device=edge_index.device)
    node_set[node_idx] = True
    mask = node_set[edge_index[0]] & node_set[edge_index[1]]
    sub_edges = edge_index[:, mask]
    remap = torch.full((num_nodes,), -1, dtype=torch.long, device=edge_index.device)
    remap[node_idx] = torch.arange(batch_size, device=edge_index.device)
    sub_edges = remap[sub_edges]
    return node_idx, sub_edges


if USE_SUBGRAPH_BATCHING:
    print("Subgraph mini-batch training is enabled. Re-initialize model/optimizer before running.")
else:
    print("Subgraph mini-batch training disabled (full-graph mode).")

## 8. Investigation: loss stability, cluster purity, KNN cache

Run after section 7. Diagnoses the three improvement areas from the first Colab run.

In [ ]:
# --- 8a. Loss stability: raw vs clamped quadrance on current embeddings ---
model.eval()
with torch.no_grad():
    z = model(train_features, train_edge_index)

    def raw_quadrance(a, b, eps=1e-9):
        dot = uhg_inner_product(a, b)
        denom = torch.clamp((uhg_norm(a) * uhg_norm(b)).abs(), min=eps)
        return 1 - (dot ** 2) / denom

    sample_edges = train_edge_index[:, : min(50000, train_edge_index.size(1))]
    raw_q = raw_quadrance(z[sample_edges[0]], z[sample_edges[1]])
    clamp_q = uhg_quadrance(z[sample_edges[0]], z[sample_edges[1]])

    print("Quadrance on sample edges (current checkpoint):")
    print(f"  raw:   min={raw_q.min():.3f}, max={raw_q.max():.3f}, mean={raw_q.mean():.3f}")
    print(f"  clamp: min={clamp_q.min():.3f}, max={clamp_q.max():.3f}, mean={clamp_q.mean():.3f}")
    print(f"  fraction raw < 0: {(raw_q < 0).float().mean():.1%}")
    print(
        "  spread term swing (0.1 * mean): "
        f"{0.1 * raw_q.mean():.3f} raw vs {0.1 * clamp_q.mean():.3f} clamped"
    )

    parts_fixed = uhg_loss(
        z, val_edge_index, cfg.spread_weight, cfg.neg_multiplier,
        generator=val_loss_generator, return_parts=True,
    )
    totals = []
    for seed in range(5):
        g = torch.Generator(device=device).manual_seed(seed)
        p = uhg_loss(
            z, val_edge_index, cfg.spread_weight, cfg.neg_multiplier,
            generator=g, return_parts=True,
        )
        totals.append(float(p["total_loss"]))
    print("\nVal total loss with 5 different neg-sample seeds (same embeddings):")
    print(f"  min={min(totals):.4f}, max={max(totals):.4f}, spread={max(totals)-min(totals):.4f}")
    print(
        f"  fixed-seed val (seed=42): total={float(parts_fixed['total_loss']):.4f}, "
        f"link={float(parts_fixed['link_loss']):.4f}"
    )
    print("  -> Explains spurious best_val=-7.07 in the first run.")

In [ ]:
# --- 8b. Cluster purity: DBSCAN sweep + cluster 54 profile ---

def cluster_purity_table(true_labels, cluster_labels, target_label: int) -> pd.DataFrame:
    rows = []
    for cid in np.unique(cluster_labels):
        mask = cluster_labels == cid
        size = int(mask.sum())
        attack_count = int((true_labels[mask] == target_label).sum())
        attack_rate = attack_count / size if size else 0.0
        rows.append({"cluster": int(cid), "size": size, "attack_count": attack_count, "attack_rate": attack_rate})
    df = pd.DataFrame(rows)
    baseline = (true_labels == target_label).mean()
    df["enrichment_vs_baseline"] = df["attack_rate"] / baseline if baseline else np.nan
    return df.sort_values("attack_rate", ascending=False)


def score_dbscan_config(labels, true_labels, target_label: int) -> dict:
    df = cluster_purity_table(true_labels, labels, target_label)
    df_nonoise = df[df["cluster"] != -1]
    if df_nonoise.empty:
        return {"top_attack_rate": 0.0, "top_cluster": -1, "n_clusters": 0, "noise_frac": 1.0}
    top = df_nonoise.iloc[0]
    return {
        "top_attack_rate": float(top["attack_rate"]),
        "top_cluster": int(top["cluster"]),
        "top_size": int(top["size"]),
        "top_attack_count": int(top["attack_count"]),
        "n_clusters": int((df["cluster"] != -1).sum()),
        "noise_frac": float((labels == -1).mean()),
    }


if "source_ip" not in train_df.columns:
    print("source_ip not available; skip cluster purity analysis")
else:
    train_labels = train_df["source_ip"].to_numpy()
    print(f"Baseline attack rate (source_ip=={cfg.attack_label}): {(train_labels == cfg.attack_label).mean():.1%}")

    current = cluster_purity_table(train_labels, dbscan_umap_labels, cfg.attack_label)
    print("\nTop UMAP clusters by attack rate (current eps=0.5, min_samples=5):")
    print(current.head(10).to_string(index=False))

    c54 = current[current["cluster"] == 54]
    if not c54.empty:
        row = c54.iloc[0]
        print(
            f"\nCluster 54: size={row['size']}, attacks={row['attack_count']} "
            f"({row['attack_rate']:.1%}), enrichment={row['enrichment_vs_baseline']:.2f}x"
        )

    sweep_rows = []
    for eps in [0.3, 0.4, 0.5, 0.6, 0.8, 1.0]:
        for ms in [3, 5, 10, 15]:
            labels = apply_dbscan(umap_embeddings, eps, ms)
            sweep_rows.append({"eps": eps, "min_samples": ms, **score_dbscan_config(labels, train_labels, cfg.attack_label)})
    sweep_df = pd.DataFrame(sweep_rows).sort_values(["top_attack_rate", "top_attack_count"], ascending=False)
    print("\nDBSCAN grid search (top 10 by attack purity):")
    print(sweep_df.head(10).to_string(index=False))

    best = sweep_df.iloc[0]
    print(
        f"\nSuggested: eps={best['eps']}, min_samples={int(best['min_samples'])} "
        f"-> cluster {int(best['top_cluster'])} at {best['top_attack_rate']:.1%} attack rate"
    )

In [ ]:
# --- 8c. KNN cache verification ---
print(f"REBUILD_KNN={cfg.REBUILD_KNN}")
for split_name in ("train", "val", "test"):
    meta_path = os.path.join(cfg.ARTIFACT_DIR, f"{split_name}_edge_index.meta.json")
    cache_path = os.path.join(cfg.ARTIFACT_DIR, f"{split_name}_edge_index.pt")
    print(f"\n{split_name}:")
    print(f"  cache exists: {os.path.exists(cache_path)}")
    print(f"  meta exists:  {os.path.exists(meta_path)}")
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            print(f"  metadata: {json.load(f)}")
    if os.path.exists(cache_path):
        ei = torch.load(cache_path, map_location="cpu")
        print(f"  edges: {ei.shape[1]}")

print("\nRe-run section 3 with REBUILD_KNN=False to confirm sub-second cache loads.")